# FemScan AI — Cervical Classifier Training

**Before you start:** Runtime > Change runtime type > **T4 GPU**

### What this notebook does
1. Clones the repo and installs dependencies
2. Mounts Google Drive (where you upload your images)
3. Copies images into the right folder structure
4. Runs `train_cervical.py` with class-weighted loss + balanced sampling
5. Saves the best checkpoint back to your Drive

### One-time Drive setup (do this before running)
Zip your `data/sipakmed/` folder locally and upload it to Google Drive:
```
# On your machine (Windows PowerShell):
cd "C:\Users\ADMIN\OneDrive\Desktop\OSBORN\HACKATHON POSTINGS\femscan-ai"
Compress-Archive -Path data\sipakmed -DestinationPath sipakmed_images.zip
```
Upload `sipakmed_images.zip` to the root of your Google Drive.

## Step 1 — GPU check

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Step 2 — Clone repo + install dependencies

In [ ]:
import os

REPO_URL    = 'https://github.com/Donitero/CerviScan.git'
BRANCH      = 'claude/bold-heyrovsky'
REPO_DIR    = '/content/femscan-ai'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print('Repo ready at', REPO_DIR)

In [ ]:
# Install all requirements (torch/torchvision are pre-installed on Colab,
# but we pin the rest to match requirements.txt)
!pip install -q \
    timm>=0.9.12 \
    albumentations>=1.3.1 \
    opencv-python-headless>=4.8.0 \
    scikit-learn>=1.3.0 \
    pandas>=2.1.0 \
    pytorch-grad-cam>=1.5.0

print('Dependencies installed.')

## Step 3 — Mount Drive and unzip images

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
import zipfile, shutil, os
from pathlib import Path

ZIP_PATH  = '/content/drive/MyDrive/sipakmed_images.zip'   # <-- adjust if you put it in a subfolder
DEST      = f'{REPO_DIR}/data/sipakmed'

if not Path(ZIP_PATH).exists():
    raise FileNotFoundError(
        f'Could not find {ZIP_PATH}\n'
        'Upload sipakmed_images.zip to the root of your Google Drive first.'
    )

print(f'Unzipping {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(f'{REPO_DIR}/data')

# Count what we have
classes = [d for d in Path(DEST).iterdir() if d.is_dir()]
for c in sorted(classes):
    print(f'  {c.name}: {len(list(c.glob("*.png")))} images')
print('Done.')

## Step 4 — Rebuild split manifest

The manifest CSV in the repo uses Windows paths. We regenerate it here so paths point to `/content/femscan-ai/...`.

In [ ]:
!python scripts/ingest_unclean_data.py 2>/dev/null || python - << 'PYEOF'
# Fallback: rebuild manifest directly from data/sipakmed/ without needing unclean_data/
import csv
from pathlib import Path
from sklearn.model_selection import train_test_split

PROJ     = Path('/content/femscan-ai')
SRC      = PROJ / 'data' / 'sipakmed'
OUT_CSV  = PROJ / 'data' / 'sipakmed_split.csv'
SEED     = 42

rows = []
for cls_dir in sorted(SRC.iterdir()):
    if not cls_dir.is_dir(): continue
    imgs = sorted(cls_dir.glob('*.png'))
    n = len(imgs)
    if n < 3:
        for p in imgs: rows.append({'filepath': str(p.relative_to(PROJ)), 'class': cls_dir.name, 'split': 'train'})
        continue
    tr, tmp = train_test_split(imgs, test_size=0.30, random_state=SEED)
    va, te  = train_test_split(tmp,  test_size=0.50, random_state=SEED) if len(tmp) >= 2 else (tmp, [])
    for p in tr: rows.append({'filepath': str(p.relative_to(PROJ)), 'class': cls_dir.name, 'split': 'train'})
    for p in va: rows.append({'filepath': str(p.relative_to(PROJ)), 'class': cls_dir.name, 'split': 'val'})
    for p in te: rows.append({'filepath': str(p.relative_to(PROJ)), 'class': cls_dir.name, 'split': 'test'})

with open(OUT_CSV, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['filepath','class','split'])
    w.writeheader(); w.writerows(rows)

from collections import Counter
splits = Counter(r['split'] for r in rows)
print(f'Manifest rebuilt: {len(rows)} rows  train={splits["train"]} val={splits["val"]} test={splits["test"]}')
print(f'Saved to {OUT_CSV}')
PYEOF

## Step 5 — Verify dataset loads correctly

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from scripts.train_cervical import (
    CervicalDataset, build_train_transform, build_val_transform,
    compute_class_weights, compute_sample_weights
)
from models.cervical_classifier import CLASS_NAMES
import torch

manifest = f'{REPO_DIR}/data/sipakmed_split.csv'
train_ds = CervicalDataset(manifest, 'train', build_train_transform(224))
val_ds   = CervicalDataset(manifest, 'val',   build_val_transform(224))

print(f'Train: {len(train_ds)} samples')
print(f'Val  : {len(val_ds)} samples')
print('\nClass distribution (train):')
for cls, n in sorted(train_ds.class_counts().items()):
    print(f'  {cls:<30}: {n}')

# Show class weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = compute_class_weights(train_ds, device)
print('\nClass weights for loss:')
for cls, w in zip(CLASS_NAMES, weights.tolist()):
    bar = '#' * int(w * 5)
    print(f'  {cls:<30}: {w:.2f}  {bar}')

In [ ]:
# Quick sanity check: load one batch and visualise
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

loader = DataLoader(train_ds, batch_size=8, shuffle=True)
imgs, labels = next(iter(loader))

# De-normalise for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
imgs_display = (imgs * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs_display[i].permute(1,2,0).numpy())
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Augmented training samples', fontsize=12)
plt.tight_layout()
plt.show()
print('Batch shape:', imgs.shape)

## Step 6 — Train

Default config: **25 epochs**, freeze backbone for first 5, then fine-tune.
On a T4 GPU this takes ~8-12 minutes for 232 images.

Adjust `EPOCHS` and `BATCH_SIZE` as needed.

In [ ]:
EPOCHS       = 25
BATCH_SIZE   = 16
FREEZE_EPOCHS = 5
LR           = 3e-4

!python {REPO_DIR}/scripts/train_cervical.py \
    --manifest  {REPO_DIR}/data/sipakmed_split.csv \
    --out-dir   {REPO_DIR}/trained_models \
    --epochs    {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --freeze-epochs {FREEZE_EPOCHS} \
    --lr        {LR}

## Step 7 — Save checkpoint to Google Drive

In [ ]:
import shutil, time
from pathlib import Path

src  = Path(f'{REPO_DIR}/trained_models/cervical_best.pt')
tag  = time.strftime('%Y%m%d_%H%M')
dest = Path(f'/content/drive/MyDrive/femscan_checkpoints/cervical_best_{tag}.pt')
dest.parent.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copy2(src, dest)
    print(f'Checkpoint saved to Drive: {dest}')
else:
    print('No checkpoint found — did training complete?')

## Step 8 — Evaluate on test set

In [ ]:
import torch
import sys
sys.path.insert(0, REPO_DIR)

from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from scripts.train_cervical import CervicalDataset, build_val_transform
from models.cervical_classifier import CervicalClassifier, CLASS_NAMES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best checkpoint
ckpt_path = f'{REPO_DIR}/trained_models/cervical_best.pt'
model = CervicalClassifier(pretrained=False, checkpoint_path=ckpt_path).to(device)
model.eval()
print('Loaded checkpoint from', ckpt_path)

# Test set
test_ds = CervicalDataset(
    f'{REPO_DIR}/data/sipakmed_split.csv', 'test', build_val_transform(224)
)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print('\nClassification Report:')
print(classification_report(
    all_labels, all_preds,
    target_names=CLASS_NAMES,
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/femscan_checkpoints/confusion_matrix.png', dpi=150)
plt.show()